# Import RDKit, install if not available

In [ ]:

try:
  import rdkit
except ImportError:
  !pip install rdkit
  import rdkit

import warnings

# In the list add proper known monomners for testing reactions

In [ ]:
sample_smiles = [
            "C=C"       
            ]

# Validate SMILES strings

In [ ]:
from rdkit import Chem
for s in sample_smiles:
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        raise ValueError(f"SMILES '{s}' is invalid (RDKit could not parse it).")


# Define your required monomers functionality 

In [ ]:
monomers = {
        'vinyl_monomer': {
        'functionality_type': 'vinyl',
        'smarts_1': '[CH2]=[C;!R]',
        'group_name': 'vinyl',
        'comments': None
    },
}


# Run validation checks on monomer functionality definitions

In [ ]:

for funct in monomers.values():
    functionality_type = funct["functionality_type"]
    if functionality_type is None:
        raise ValueError("Functionality type is missing.")
    if functionality_type not in ["di_identical", "di_different", "mono", "vinyl"]:
        raise ValueError(f"Functionality type '{functionality_type}' is invalid.")
    if "smarts_1" not in funct:
        raise ValueError("Missing 'smarts_1' must be provided.")
    smart_1 = funct.get("smarts_1")
    if Chem.MolFromSmarts(smart_1) is None:
        raise ValueError(f"'smarts_1' '{smart_1}' is invalid.")
    if functionality_type == "di_different":
        smart_2 = funct.get("smarts_2")
        if smart_2 is None:
            raise ValueError("Missing 'smarts_2' for 'di_different' functionality type.")
        if Chem.MolFromSmarts(smart_2) is None:
            raise ValueError(f"'smarts_2' '{smart_2}' is invalid.")
    if "group_name" not in funct:
        raise ValueError("Missing 'group_name' must be provided.")
    group_name = funct.get("group_name")
    if not isinstance(group_name, str) or not group_name.strip():
        raise ValueError(f"'group_name' '{group_name}' is invalid.")
    print(f"{group_name} definitions are valid.")
    

# Testing Function Block

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw

def validate_functional_group(sample_smiles: list[str], monomers: dict[str, dict[str, str]]):
    results = []

    for monomer_1_smiles in sample_smiles:
        mol_1 = Chem.MolFromSmiles(monomer_1_smiles)
        if mol_1 is None:
            print(f"Skipping invalid SMILES: {monomer_1_smiles}")
            continue

        # make explicit Hs (MUST assign)
        mol_1 = Chem.AddHs(mol_1)

        for funtionality in monomers.values():

            if funtionality["functionality_type"] == "di_different":
                smarts_1 = Chem.MolFromSmarts(funtionality["smarts_1"])
                smarts_2 = Chem.MolFromSmarts(funtionality["smarts_2"])
                if smarts_1 is None or smarts_2 is None:
                    continue

                if mol_1.HasSubstructMatch(smarts_1) and mol_1.HasSubstructMatch(smarts_2):
                    print(f"Monomer {monomer_1_smiles} matches functionality {funtionality['group_name']}")
                    print("Different functional passed.")

                    # get *all* matches (not just first) so highlight is complete
                    matches_1 = mol_1.GetSubstructMatches(smarts_1)
                    matches_2 = mol_1.GetSubstructMatches(smarts_2)

                    atoms_to_highlight = []
                    for m in matches_1:
                        atoms_to_highlight.extend(list(m))
                    for m in matches_2:
                        atoms_to_highlight.extend(list(m))

                    # dedupe + keep order-ish
                    atoms_to_highlight = list(dict.fromkeys(atoms_to_highlight))

                    results.append((mol_1, atoms_to_highlight, funtionality["group_name"]))

            if funtionality["functionality_type"] == "di_identical":
                smarts_1 = Chem.MolFromSmarts(funtionality["smarts_1"])
                if smarts_1 is None:
                    continue

                matches = mol_1.GetSubstructMatches(smarts_1)
                if len(matches) >= 2:
                    print(f"Monomer {monomer_1_smiles} matches functionality {funtionality['group_name']}")
                    print("Identical functional passed.")

                    atoms_to_highlight = []
                    for match in matches:
                        atoms_to_highlight.extend(list(match))
                    atoms_to_highlight = list(dict.fromkeys(atoms_to_highlight))

                    results.append((mol_1, atoms_to_highlight, funtionality["group_name"]))

            if funtionality["functionality_type"] in {"vinyl", "mono"}:
                smarts_1 = Chem.MolFromSmarts(funtionality["smarts_1"])

                if smarts_1 is None:
                    continue

                matches = mol_1.GetSubstructMatches(smarts_1)

                if matches:
                    print(
                        f"Monomer {monomer_1_smiles} matches functionality "
                        f"{funtionality['group_name']}"
                    )
                    print(
                        f"{funtionality['functionality_type'].capitalize()} "
                        "functional passed."
                    )

                    atoms_to_highlight = list(
                        dict.fromkeys(
                            atom_idx
                            for match in matches
                            for atom_idx in match
                        )
                    )

                    results.append(
                        (
                            mol_1,
                            atoms_to_highlight,
                            funtionality["group_name"],
                        )
                    )
                    
    if not results:
        raise ValueError("No matching functional group found.")
    return results


# Test the function and Display results

In [ ]:


results = validate_functional_group(sample_smiles, monomers)


# Create a grid from the molecules and highlights directly
grid_img = Draw.MolsToGridImage(
    [r[0] for r in results], 
    molsPerRow=3, 
    subImgSize=(900, 900),
    highlightAtomLists=[r[1] for r in results],
    legends=[r[2] for r in results]
)

display(grid_img)

# Define reaction definitions

In [ ]:

reactions_definitions = {
        'Vinyl Addition Polymerization Initiation': {
        'same_reactants': True,
        'reactant_1': 'vinyl',
        'product': 'vinyl_chain_end_radical',
        'delete_atom': False,
        'reaction': (
            '[CH2:3]=[C;!R:1].'
            '[CH2:2]=[C;!R:4]'
            '>>'
            '[CH3:3]-[C:1]-[CH2:2]-[C;!R:4]'
        )   
        }
}

# Validate reaction SMARTS and references

In [ ]:

from rdkit import Chem
from rdkit.Chem import rdChemReactions
import json
def validate_reaction_smarts(smarts_string):
    """
    Validates a reaction SMARTS string using RDKit.
    Returns True if valid, False otherwise.
    """
    try:
        # Attempt to create a reaction object from the SMARTS string
        rxn = rdChemReactions.ReactionFromSmarts(smarts_string)
        
        # If the above line runs without error, the SMARTS is syntactically valid
        if rxn is not None:
            print(f"'{smarts_string}' is a valid reaction SMARTS.")
            return True
        else:
            # Handle cases where no exception is raised, but the object is None
            print(f"'{smarts_string}' is an invalid reaction SMARTS (returned None).")
            return False
    except Exception as e:
        # If an exception is caught, the SMARTS is invalid
        print(f"'{smarts_string}' is an invalid reaction SMARTS.")
        # Optional: print the error message for debugging
        # print(f"Error details: {e}")
        return False
    
def are_all_atoms_mapped(reaction_smarts):
    rxn = rdChemReactions.ReactionFromSmarts(reaction_smarts)
    if rxn is None:
        return False, "Invalid reaction SMARTS"

    # Check reactants
    for reactant_mol in rxn.GetReactants():
        for atom in reactant_mol.GetAtoms():
            if atom.GetAtomMapNum() == 0:
                return False, "Unmapped atom found in reactants"
    
    # Check products
    for product_mol in rxn.GetProducts():
        for atom in product_mol.GetAtoms():
            if atom.GetAtomMapNum() == 0:
                return False, "Unmapped atom found in products"

    return True, "All atoms are mapped"

for reaction in reactions_definitions.values():
    reaction_smarts = reaction.get("reaction")
    same_reactants = reaction.get("same_reactants")
    reactant_1 = reaction.get("reactant_1")
    product = reaction.get("product")
    delete_atom = reaction.get("delete_atom")
    reference = reaction.get("reference")
    if same_reactants is None:
        raise ValueError("Same reactants flag is missing.")
    if reactant_1 is None:
        raise ValueError("Reactant 1 information is missing.")
    if not same_reactants:
        reactant_2 = reaction.get("reactant_2")
        if reactant_2 is None:
            raise ValueError("Reactant 2 information is missing for different reactants.")
    if product is None:
        raise ValueError("Product information is missing.")
    if delete_atom is None:
        raise ValueError("Delete atom flag is missing.")
    if reaction_smarts is None:
        raise ValueError("Reaction SMARTS is missing.")
    validate_reaction_smarts(reaction_smarts)
    is_mapped, message = are_all_atoms_mapped(reaction_smarts)
    if not is_mapped:
        warnings.warn(f"Reaction SMARTS '{reaction_smarts}' has unmapped atoms: {message}")


print(f"{json.dumps(reaction, indent=2)}")
print("Reaction definition is valid.")

In [ ]:
from collections import Counter


def smart_mapping(reactant, smarts_template, match_tuple):
    """
    Transfer atom-map numbers from a SMARTS template to a matched reactant.

    Only explicitly mapped SMARTS atoms are processed. Unmapped atoms are
    allowed and remain unmapped.

    Mapping rules
    -------------
    - Atom-map number 0 means "unmapped" and is ignored.
    - Every nonzero map number in the SMARTS template must be unique.
    - Each mapped SMARTS atom must correspond to exactly one matched atom
      in the reactant.
    - Existing atom-map numbers on the matched reactant atoms are replaced
      by the mapping defined by the SMARTS template.

    Parameters
    ----------
    reactant : Chem.Mol
        Reactant molecule to modify in place.

    smarts_template : Chem.Mol
        SMARTS reactant template containing atom-map numbers.

    match_tuple : tuple
        RDKit substructure match. Position i corresponds to SMARTS atom i.

    Raises
    ------
    ValueError
        If a mapped SMARTS atom has no corresponding matched atom or if
        duplicate atom-map numbers occur in the SMARTS template.
    """
    if not match_tuple:
        return

    map_numbers = [
        atom.GetAtomMapNum()
        for atom in smarts_template.GetAtoms()
        if atom.GetAtomMapNum() != 0
    ]

    duplicate_maps = sorted(
        atom_map
        for atom_map, count in Counter(map_numbers).items()
        if count > 1
    )

    if duplicate_maps:
        raise ValueError(
            "SMARTS template contains duplicate atom-map numbers: "
            f"{duplicate_maps}. "
            "Every mapped atom must have a unique map number."
        )

    for smarts_atom in smarts_template.GetAtoms():
        map_num = smarts_atom.GetAtomMapNum()

        if map_num == 0:
            continue

        smarts_index = smarts_atom.GetIdx()

        if smarts_index >= len(match_tuple):
            raise ValueError(
                f"SMARTS atom index {smarts_index} with map :{map_num} "
                "has no corresponding atom in the RDKit match."
            )

        reactant_atom_index = match_tuple[smarts_index]

        reactant_atom = reactant.GetAtomWithIdx(
            reactant_atom_index
        )

        reactant_atom.SetAtomMapNum(map_num)


def _atom_map_counts(templates) -> Counter:
    """
    Count all nonzero atom-map numbers in a collection of RDKit templates.

    Unmapped atoms (map number 0) are ignored.
    """
    counts = Counter()

    for template in templates:
        for atom in template.GetAtoms():
            atom_map = atom.GetAtomMapNum()

            if atom_map:
                counts[atom_map] += 1

    return counts


def _atom_maps_in_templates(templates) -> set[int]:
    """
    Return all nonzero atom-map numbers present in RDKit templates.
    """
    return set(_atom_map_counts(templates))


def _duplicate_atom_maps(templates) -> list[int]:
    """
    Return atom-map numbers appearing more than once in a template set.

    For AutoREACTER, each nonzero atom-map number must identify exactly
    one atom on each side of the reaction.
    """
    counts = _atom_map_counts(templates)

    return sorted(
        atom_map
        for atom_map, count in counts.items()
        if count > 1
    )


def _has_bond_between_atom_maps(
    templates,
    atom_map_1: int,
    atom_map_2: int,
) -> bool:
    """
    Return True if any template contains a bond between two atom maps.
    """
    target = {atom_map_1, atom_map_2}

    for template in templates:
        for bond in template.GetBonds():
            begin_map = (
                bond.GetBeginAtom().GetAtomMapNum()
            )

            end_map = (
                bond.GetEndAtom().GetAtomMapNum()
            )

            if {begin_map, end_map} == target:
                return True

    return False


def _validate_reaction_smarts(
    reaction_name: str,
    reaction: dict,
) -> list[str]:
    """
    Validate one AutoREACTER reaction-library entry.

    AutoREACTER atom-mapping rules
    ------------------------------
    Not every atom must be mapped.

    However, every atom that IS mapped must obey a strict 1:1 mapping:

        reactant atom :N  <->  product atom :N

    Therefore:

        - each nonzero atom-map number may occur only once across all
          reactant templates;
        - each nonzero atom-map number may occur only once across all
          product templates;
        - the mapped-atom set on the reactant side must exactly match
          the mapped-atom set on the product side.

    AutoREACTER additionally requires initiator atoms :1 and :2 by
    default. Both must exist on both sides of the reaction, and the
    product must contain a bond between them.

    Optional override
    -----------------
    A different initiator pair may be specified with:

        "initiator_atom_maps": (a, b)

    Special reactions may disable the initiator-bond requirement with:

        "validate_initiator_bond": False

    Note
    ----
    Disabling the initiator-bond check does NOT disable the general
    1:1 atom-mapping validation.
    """
    errors: list[str] = []

    smarts = reaction.get("reaction")

    if not isinstance(smarts, str) or not smarts.strip():
        return [
            f"{reaction_name}: missing or invalid required key "
            "'reaction'"
        ]

    if rdChemReactions is None:
        return [
            f"{reaction_name}: RDKit is required to validate "
            "reaction SMARTS"
        ]

    initiator_atom_maps = reaction.get(
        "initiator_atom_maps",
        (1, 2),
    )

    if (
        not isinstance(
            initiator_atom_maps,
            (list, tuple),
        )
        or len(initiator_atom_maps) != 2
    ):
        return [
            f"{reaction_name}: 'initiator_atom_maps' must contain "
            "exactly two atom-map numbers"
        ]

    try:
        initiator_1 = int(initiator_atom_maps[0])
        initiator_2 = int(initiator_atom_maps[1])

    except (TypeError, ValueError):
        return [
            f"{reaction_name}: 'initiator_atom_maps' must contain "
            "integer atom-map numbers"
        ]

    if initiator_1 <= 0 or initiator_2 <= 0:
        return [
            f"{reaction_name}: initiator atom-map numbers must be "
            "greater than zero"
        ]

    if initiator_1 == initiator_2:
        return [
            f"{reaction_name}: initiator atom maps must be different"
        ]

    try:
        rdkit_reaction = (
            rdChemReactions.ReactionFromSmarts(smarts)
        )

    except Exception as error:
        return [
            f"{reaction_name}: invalid reaction SMARTS: {error}"
        ]

    if rdkit_reaction is None:
        return [
            f"{reaction_name}: RDKit could not parse reaction SMARTS"
        ]

    num_reactants = (
        rdkit_reaction.GetNumReactantTemplates()
    )

    num_products = (
        rdkit_reaction.GetNumProductTemplates()
    )

    if num_reactants != 2:
        errors.append(
            f"{reaction_name}: AutoREACTER currently requires "
            f"exactly 2 reactant templates; found {num_reactants}"
        )

    if num_products == 0:
        errors.append(
            f"{reaction_name}: reaction SMARTS contains "
            "no product templates"
        )

    reactant_templates = [
        rdkit_reaction.GetReactantTemplate(i)
        for i in range(num_reactants)
    ]

    product_templates = [
        rdkit_reaction.GetProductTemplate(i)
        for i in range(num_products)
    ]

    # ---------------------------------------------------------
    # Validate uniqueness of mapped atoms
    # ---------------------------------------------------------

    duplicate_reactant_maps = _duplicate_atom_maps(
        reactant_templates
    )

    duplicate_product_maps = _duplicate_atom_maps(
        product_templates
    )

    if duplicate_reactant_maps:
        errors.append(
            f"{reaction_name}: duplicate atom-map numbers in "
            f"reactants: {duplicate_reactant_maps}. "
            "Each mapped atom must have a unique map number."
        )

    if duplicate_product_maps:
        errors.append(
            f"{reaction_name}: duplicate atom-map numbers in "
            f"products: {duplicate_product_maps}. "
            "Each mapped atom must have a unique map number."
        )

    # ---------------------------------------------------------
    # Get mapped-atom sets
    # ---------------------------------------------------------

    reactant_maps = _atom_maps_in_templates(
        reactant_templates
    )

    product_maps = _atom_maps_in_templates(
        product_templates
    )

    # ---------------------------------------------------------
    # Require strict 1:1 mapping
    # ---------------------------------------------------------

    missing_from_products = (
        reactant_maps - product_maps
    )

    missing_from_reactants = (
        product_maps - reactant_maps
    )

    if missing_from_products:
        errors.append(
            f"{reaction_name}: mapped atoms present in reactants "
            "but missing from products: "
            f"{sorted(missing_from_products)}. "
            "Mapped atoms must have a 1:1 reactant/product mapping."
        )

    if missing_from_reactants:
        errors.append(
            f"{reaction_name}: mapped atoms present in products "
            "but missing from reactants: "
            f"{sorted(missing_from_reactants)}. "
            "Mapped atoms must have a 1:1 reactant/product mapping."
        )

    # ---------------------------------------------------------
    # Validate mandatory initiator atoms
    # ---------------------------------------------------------

    required_maps = {
        initiator_1,
        initiator_2,
    }

    missing_reactant_initiators = (
        required_maps - reactant_maps
    )

    missing_product_initiators = (
        required_maps - product_maps
    )

    if missing_reactant_initiators:
        errors.append(
            f"{reaction_name}: required initiator atom maps "
            "missing from reactants: "
            f"{sorted(missing_reactant_initiators)}"
        )

    if missing_product_initiators:
        errors.append(
            f"{reaction_name}: required initiator atom maps "
            "missing from products: "
            f"{sorted(missing_product_initiators)}"
        )

    # ---------------------------------------------------------
    # Validate required product initiator bond
    # ---------------------------------------------------------

    validate_initiator_bond = reaction.get(
        "validate_initiator_bond",
        True,
    )

    if (
        validate_initiator_bond
        and not missing_product_initiators
    ):
        product_has_initiator_bond = (
            _has_bond_between_atom_maps(
                product_templates,
                initiator_1,
                initiator_2,
            )
        )

        if not product_has_initiator_bond:
            errors.append(
                f"{reaction_name}: product does not contain "
                "the required AutoREACTER initiator bond between "
                f"atom maps :{initiator_1} and :{initiator_2}"
            )

    return errors


def validate_reactions(reactions: dict) -> None:
    """
    Validate the complete merged AutoREACTER reaction library.

    Raises
    ------
    TypeError
        If the reaction library is not a dictionary.

    ValueError
        If one or more reaction definitions are invalid.
    """
    if not isinstance(reactions, dict):
        raise TypeError(
            "Reaction library must be a dictionary."
        )

    errors: list[str] = []

    for reaction_name, reaction in reactions.items():

        if not isinstance(reaction, dict):
            errors.append(
                f"{reaction_name}: reaction entry must be "
                "a dictionary"
            )
            continue

        errors.extend(
            _validate_reaction_smarts(
                reaction_name,
                reaction,
            )
        )

    if errors:
        message = "\n".join(
            f"  - {error}"
            for error in errors
        )

        raise ValueError(
            "Reaction library validation failed:\n"
            f"{message}"
        )


def build_reaction(rxn_smarts: str):
    """
    Create an RDKit ChemicalReaction from reaction SMARTS.

    AutoREACTER currently supports reactions containing exactly
    two reactant templates.

    Parameters
    ----------
    rxn_smarts : str
        Reaction SMARTS string.

    Returns
    -------
    rdChemReactions.ChemicalReaction
        Parsed RDKit reaction object.

    Raises
    ------
    ValueError
        If the SMARTS is empty, invalid, or does not contain exactly
        two reactant templates.
    """
    if (
        not isinstance(rxn_smarts, str)
        or not rxn_smarts.strip()
    ):
        raise ValueError(
            "Reaction SMARTS must be a non-empty string."
        )

    try:
        rxn = rdChemReactions.ReactionFromSmarts(
            rxn_smarts
        )

    except Exception as error:
        raise ValueError(
            f"Could not parse reaction SMARTS: {error}"
        ) from error

    if rxn is None:
        raise ValueError(
            "RDKit could not parse reaction SMARTS: "
            f"{rxn_smarts}"
        )

    num_reactants = rxn.GetNumReactantTemplates()

    if num_reactants != 2:
        raise ValueError(
            "AutoREACTER currently requires exactly two "
            f"reactant templates; found {num_reactants}."
        )

    if rxn.GetNumProductTemplates() == 0:
        raise ValueError(
            "Reaction SMARTS must contain at least one "
            "product template."
        )

    return rxn


def _clear_atom_maps(mol):
    """
    Remove existing atom-map numbers from an RDKit molecule.

    Reaction mapping in AutoREACTER is defined by the reaction SMARTS,
    not by map numbers that may already exist in the input SMILES.
    """
    for atom in mol.GetAtoms():
        atom.SetAtomMapNum(0)


def _build_molecule(smiles: str):
    """
    Build an RDKit molecule from SMILES and add explicit hydrogens.

    Existing atom-map numbers from the input SMILES are cleared because
    AutoREACTER derives reaction mapping from the reaction SMARTS.
    """
    if (
        not isinstance(smiles, str)
        or not smiles.strip()
    ):
        raise ValueError(
            "Reactant SMILES must be a non-empty string."
        )

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(
            "RDKit could not parse reactant SMILES: "
            f"{smiles}"
        )

    mol = Chem.AddHs(mol)

    _clear_atom_maps(mol)

    return mol


def build_reactants(
    reactant_smiles_1: str,
    reactant_smiles_2: str | None = None,
):
    """
    Build two independent RDKit reactant molecules.

    Explicit hydrogens are added to both molecules.

    If reactant_smiles_2 is None, a second independent molecule is
    constructed from reactant_smiles_1. This is important for
    self-reactions because A + A must use two separate RDKit Mol
    objects.

    Returns
    -------
    tuple
        (mol_reactant_1, mol_reactant_2)
    """
    if reactant_smiles_2 is None:
        reactant_smiles_2 = reactant_smiles_1

    mol_reactant_1 = _build_molecule(
        reactant_smiles_1
    )

    mol_reactant_2 = _build_molecule(
        reactant_smiles_2
    )

    return (
        mol_reactant_1,
        mol_reactant_2,
    )


def reaction_tuples(
    same_reactants: bool,
    mol_reactant_1,
    mol_reactant_2,
):
    """
    Generate possible reactant orderings.

    Self-reaction
    -------------
        A + A

    Different reactants
    -------------------
        A + B
        B + A

    Fresh molecule copies are generated for every attempt so atom maps
    applied during one attempt cannot contaminate another.
    """
    if same_reactants:
        return [
            (
                Chem.Mol(mol_reactant_1),
                Chem.Mol(mol_reactant_2),
            )
        ]

    return [
        (
            Chem.Mol(mol_reactant_1),
            Chem.Mol(mol_reactant_2),
        ),
        (
            Chem.Mol(mol_reactant_2),
            Chem.Mol(mol_reactant_1),
        ),
    ]


def map_reactant_atoms(
    rxn_smarts: str,
    same_reactants: bool,
    reactant_smiles_1: str,
    reactant_smiles_2: str | None = None,
):
    """
    Match reaction SMARTS to reactants and transfer atom-map numbers.

    Not every atom is required to be mapped. Only atoms explicitly
    mapped in the SMARTS receive atom-map numbers.

    AutoREACTER requires reaction mapping itself to be valid before this
    function is called:

        - mapped atoms must have unique map numbers;
        - mapped atoms must correspond 1:1 between reactants/products;
        - initiator atom maps :1 and :2 must exist;
        - :1 and :2 must form the product initiator bond.

    For different reactants, both reactant orderings are attempted:

        A + B
        B + A

    Parameters
    ----------
    rxn_smarts : str
        Reaction SMARTS string.

    same_reactants : bool
        True for A + A reactions.

    reactant_smiles_1 : str
        SMILES of the first reactant.

    reactant_smiles_2 : str, optional
        SMILES of the second reactant.

    Returns
    -------
    tuple
        (
            [(mapped_reactant_1, mapped_reactant_2)],
            products,
        )

    Raises
    ------
    ValueError
        If no valid SMARTS match or product can be generated.
    """
    if (
        not same_reactants
        and not reactant_smiles_2
    ):
        raise ValueError(
            "reactant_smiles_2 is required when "
            "same_reactants=False."
        )

    rxn = build_reaction(
        rxn_smarts
    )

    mol_reactant_1, mol_reactant_2 = (
        build_reactants(
            reactant_smiles_1,
            reactant_smiles_1
            if same_reactants
            else reactant_smiles_2,
        )
    )

    smarts_reactant_1 = (
        rxn.GetReactantTemplate(0)
    )

    smarts_reactant_2 = (
        rxn.GetReactantTemplate(1)
    )

    matched_any_order = False

    for mol_1, mol_2 in reaction_tuples(
        same_reactants,
        mol_reactant_1,
        mol_reactant_2,
    ):

        matches_1 = mol_1.GetSubstructMatches(
            smarts_reactant_1
        )

        matches_2 = mol_2.GetSubstructMatches(
            smarts_reactant_2
        )

        if not matches_1 or not matches_2:
            continue

        matched_any_order = True

        # Current AutoREACTER behavior:
        # use the first matching reactive site.
        match_1 = matches_1[0]
        match_2 = matches_2[0]

        smart_mapping(
            mol_1,
            smarts_reactant_1,
            match_1,
        )

        smart_mapping(
            mol_2,
            smarts_reactant_2,
            match_2,
        )

        products = rxn.RunReactants(
            (
                mol_1,
                mol_2,
            )
        )

        if not products:
            # SMARTS matched this ordering, but RDKit could
            # not generate a product. Try the next ordering.
            continue

        print(
            "[INFO] Reaction SMARTS matched reactants."
        )

        print(
            "[INFO] Atom mapping applied successfully."
        )

        print(
            "[INFO] Reaction produced products."
        )

        return (
            [(mol_1, mol_2)],
            products,
        )

    if not matched_any_order:
        raise ValueError(
            "No valid substructure match was found for the "
            "supplied reactants and reaction SMARTS."
        )

    raise ValueError(
        "Reaction SMARTS matched the supplied reactants, "
        "but RDKit could not generate products for any "
        "valid reactant ordering."
    )

# Test the mapping function with reaction definitions

In [ ]:
validate_reactions(reactions_definitions)

print("[INFO] Reaction library validation passed.")


for reaction_name, test in reactions_definitions.items():

    print(f"\n[INFO] Testing reaction: {reaction_name}")

    rxn_smarts = test["reaction"]
    same_reactants = test["same_reactants"]

    reactant_smiles_1 = sample_smiles[0]

    if same_reactants:
        reactant_smiles_2 = None
    else:
        reactant_smiles_2 = sample_smiles[1]

    mapped_reactants, products = map_reactant_atoms(
        rxn_smarts=rxn_smarts,
        same_reactants=same_reactants,
        reactant_smiles_1=reactant_smiles_1,
        reactant_smiles_2=reactant_smiles_2,
    )

    print(
        f"[INFO] Reaction produced "
        f"{len(products)} product set(s)."
    )


    reactant_molecules = [
        mol
        for pair in mapped_reactants
        for mol in pair
    ]

    highlight_atoms = [
        [
            atom.GetIdx()
            for atom in mol.GetAtoms()
            if atom.GetAtomMapNum() != 0
        ]
        for mol in reactant_molecules
    ]

    img = Draw.MolsToGridImage(
        reactant_molecules,
        molsPerRow=2,
        subImgSize=(600, 600),
        highlightAtomLists=highlight_atoms,
        legends=[
            f"{reaction_name} - Reactant {i + 1}"
            for i in range(len(reactant_molecules))
        ],
    )

    display(img)

# Visualize products

In [ ]:

for product_set in products:
    img = Draw.MolsToGridImage(
        product_set,
        molsPerRow=2,
        subImgSize=(900, 900),
    )
    display(img)

: 